# Decorators
**Module 01-02 | Core Language | Day 1**

Topics covered:
- What a decorator is and how Python applies it
- `functools.wraps` — why you always need it
- Parameterized decorators (the extra wrapper layer)
- Stacking decorators
- Real DE use cases: timing, retry, caching

---
## 1. What Is a Decorator?

A decorator is just a **function that takes a function and returns a function**.

`@decorator` syntax is pure syntactic sugar — Python rewrites it before execution.

In [ ]:
# These two are exactly identical — @ is just shorthand

def shout(fn):
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        return str(result).upper()
    return wrapper

# Longhand:
def greet(name):
    return f"hello {name}"

greet = shout(greet)        # decorator applied manually
print(greet("world"))       # HELLO WORLD

# Shorthand (@ syntax):
@shout
def greet2(name):
    return f"hello {name}"

print(greet2("world"))      # HELLO WORLD — identical

In [ ]:
# The problem without functools.wraps — metadata is lost

def shout_broken(fn):
    def wrapper(*args, **kwargs):
        return str(fn(*args, **kwargs)).upper()
    return wrapper

@shout_broken
def greet(name):
    """Says hello to someone."""
    return f"hello {name}"

print(greet.__name__)   # wrapper   ← WRONG — original name is gone
print(greet.__doc__)    # None      ← WRONG — docstring is gone
# Stack traces will say 'wrapper' instead of 'greet' — very confusing

---
## 2. `functools.wraps` — Always Use It

`@functools.wraps(fn)` copies `__name__`, `__doc__`, `__module__`, and other metadata
from the original function onto the wrapper.

Without it: `help()`, logging, stack traces, and introspection tools all show the wrong name.

In [ ]:
import functools

def shout(fn):
    @functools.wraps(fn)        # copies metadata from fn onto wrapper
    def wrapper(*args, **kwargs):
        return str(fn(*args, **kwargs)).upper()
    return wrapper

@shout
def greet(name):
    """Says hello to someone."""
    return f"hello {name}"

print(greet.__name__)   # greet   ✓
print(greet.__doc__)    # Says hello to someone.   ✓
print(greet("world"))   # HELLO WORLD

---
## 3. Practical Decorator: Timing

Timing is the simplest real-world decorator — wrap any function to log how long it takes.

In [ ]:
import time
import functools

def timer(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[timer] {fn.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_query(n):
    time.sleep(0.1)
    return list(range(n))

result = slow_query(1000)
print(f"got {len(result)} rows")

---
## 4. Parameterized Decorators — The Extra Wrapper Layer

When your decorator needs arguments (e.g. `@retry(times=3)`), you need **3 levels** of nesting:

```
decorator_factory(arg)   →   decorator(fn)   →   wrapper(*args, **kwargs)
```

The outer function receives the config, the middle function receives the function, the inner function does the work.

In [ ]:
import functools

def retry(times=3, exceptions=(Exception,)):
    """Retry a function up to `times` attempts on given exception types."""
    def decorator(fn):                          # level 2: receives the function
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):           # level 3: does the actual work
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as e:
                    print(f"[retry] {fn.__name__} attempt {attempt}/{times} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator                            # level 1: returns the decorator

# Usage
call_count = 0

@retry(times=3, exceptions=(ValueError,))
def flaky_api_call():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ValueError("service unavailable")
    return "success"

print(flaky_api_call())

In [ ]:
# Why 3 levels? Trace what Python does with @retry(times=3)

# Step 1: Python evaluates retry(times=3)  → returns `decorator`
# Step 2: Python applies decorator(flaky_api_call) → returns `wrapper`
# Step 3: flaky_api_call is now bound to wrapper

# Compare with a plain decorator (2 levels):
# Step 1: Python applies timer(slow_query) → returns wrapper directly

# The () in @retry() is what triggers the extra level.

---
## 5. Stacking Decorators

You can apply multiple decorators to one function. They apply **bottom-up** — the decorator closest to `def` runs first.

In [ ]:
import functools, time

def timer(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        print(f"[timer] {fn.__name__} took {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

def retry(times=3):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    if attempt == times:
                        raise
        return wrapper
    return decorator

@timer            # applied second (outermost)
@retry(times=2)   # applied first (innermost, closest to def)
def fetch_data():
    time.sleep(0.05)
    return "data"

# Execution order: timer wraps retry(fetch_data)
# timer sees: one call that may internally retry
# So timer measures TOTAL time including retries
print(fetch_data())

In [ ]:
# Order matters — swapping changes behaviour

@retry(times=2)   # outermost: retries the timed call
@timer            # innermost: times each individual attempt
def fetch_data2():
    time.sleep(0.05)
    return "data"

# Now timer measures each attempt individually
# retry wraps the already-timed function
print(fetch_data2())

---
## 6. DE Use Cases

### 6a. Caching with `functools.lru_cache`

Python's built-in memoization — results of expensive calls are cached by their arguments.

In [ ]:
import functools

@functools.lru_cache(maxsize=128)
def fetch_exchange_rate(currency: str) -> float:
    """Simulate an expensive API call."""
    print(f"  [API call] fetching rate for {currency}")
    import time; time.sleep(0.1)   # simulated latency
    return {"EUR": 0.92, "GBP": 0.79, "JPY": 153.4}[currency]

# First call: hits the API
print(fetch_exchange_rate("EUR"))
# Second call: returns cached result instantly
print(fetch_exchange_rate("EUR"))
print(fetch_exchange_rate.cache_info())

### 6b. Logging / Auditing Decorator

Attach structured logging to any function without polluting its body — common in production pipelines.

In [ ]:
import functools, logging, time

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger(__name__)

def log_call(fn):
    """Log function name, args, return value, and duration."""
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        log.info(f"START {fn.__name__} args={args} kwargs={kwargs}")
        start = time.perf_counter()
        try:
            result = fn(*args, **kwargs)
            log.info(f"OK    {fn.__name__} elapsed={time.perf_counter()-start:.3f}s result={result!r}")
            return result
        except Exception as e:
            log.error(f"FAIL  {fn.__name__} elapsed={time.perf_counter()-start:.3f}s error={e}")
            raise
    return wrapper

@log_call
def load_partition(date: str, table: str) -> int:
    """Simulate loading a partition."""
    return 42_000   # rows loaded

load_partition("2026-04-18", "events")

### 6c. Rate-Limit / Throttle Decorator

Enforce a minimum gap between calls — useful when hitting APIs with rate limits.

In [ ]:
import functools, time

def throttle(calls_per_second: float):
    """Ensure at most `calls_per_second` calls are made per second."""
    min_interval = 1.0 / calls_per_second
    last_called = [0.0]   # mutable container so inner function can write to it

    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            elapsed = time.perf_counter() - last_called[0]
            wait = min_interval - elapsed
            if wait > 0:
                time.sleep(wait)
            last_called[0] = time.perf_counter()
            return fn(*args, **kwargs)
        return wrapper
    return decorator

@throttle(calls_per_second=2)   # max 2 calls/sec
def hit_api(page: int):
    return f"page {page} data"

start = time.perf_counter()
for i in range(4):
    print(hit_api(i))
print(f"total: {time.perf_counter() - start:.2f}s")   # ~1.5s for 4 calls at 2/sec

---
## 7. Quick Reference

| Pattern | Structure | When to use |
|---|---|---|
| Basic decorator | `def d(fn): def wrapper(...): ...; return wrapper` | Simple wrapping, no config |
| With `wraps` | Add `@functools.wraps(fn)` on wrapper | Always — preserve metadata |
| Parameterized | `def d(arg): def decorator(fn): def wrapper(...): ...` | Decorator needs config |
| Stacked | `@d1` then `@d2` above `def` | Multiple concerns; bottom-up application |
| `lru_cache` | `@functools.lru_cache(maxsize=N)` | Memoize pure functions with hashable args |

**Stacking order rule:**
```
@outermost    # applied last, runs first when the function is called
@innermost    # applied first, closest to the original function
def fn(): ...
```

**DE rule of thumb:**
- Use decorators for **cross-cutting concerns**: timing, logging, retry, caching, throttling
- Keep the function body focused on business logic only — decorators handle the plumbing

---
## Exercises

1. Write a `@validate_args` decorator that raises `TypeError` if any argument is `None`.
2. Write a `@retry(times=N, backoff=2.0)` that doubles the wait between each retry attempt (exponential backoff).
3. Stack `@log_call` and `@retry(times=3)` on a function that randomly fails — verify the log shows each retry.
4. Use `functools.lru_cache` on a recursive Fibonacci function and compare speed with/without it using `@timer`.